In [0]:
from pyspark.sql import functions as F

fact_transactions = (
    spark.table("fintech_fraud_risk.silver.transactions")
    .withColumn("date_key", F.date_format("transaction_timestamp", "yyyyMMdd").cast("int"))
    .select(
        "transaction_id", "customer_id", "merchant_id", "device_id", "date_key",
        "transaction_timestamp", "amount", "currency", "payment_method",
        "transaction_type", "transaction_status", "location"
    )
)
fact_transactions.write.format("delta").mode("overwrite").saveAsTable("fintech_fraud_risk.gold.fact_transactions")
print(f"fact_transactions: {fact_transactions.count():,} rows")

In [0]:
bronze_total = spark.table("fintech_fraud_risk.bronze.bronze_transactions").count() + spark.table("fintech_fraud_risk.bronze.bronze_transaction_cdc").count()
silver_total = spark.table("fintech_fraud_risk.silver.transactions").count()
print(f"transactions    Bronze {bronze_total:>8,} -> Silver {silver_total:>8,}  (pass rate {silver_total/bronze_total*100:.2f}%)")

In [0]:
from pyspark.sql import functions as F

# dim_customer
dim_customer = spark.table("fintech_fraud_risk.silver.customers")
dim_customer.write.format("delta").mode("overwrite").saveAsTable("fintech_fraud_risk.gold.dim_customer")

# dim_merchant
dim_merchant = spark.table("fintech_fraud_risk.silver.merchants")
dim_merchant.write.format("delta").mode("overwrite").saveAsTable("fintech_fraud_risk.gold.dim_merchant")

# dim_device
dim_device = spark.table("fintech_fraud_risk.silver.devices")
dim_device.write.format("delta").mode("overwrite").saveAsTable("fintech_fraud_risk.gold.dim_device")

# dim_date — spans your full transaction range (5 years back to today)
dim_date = (
    spark.sql("SELECT explode(sequence(to_date('2021-09-20'), to_date('2026-09-20'), interval 1 day)) AS full_date")
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))
)
dim_date.write.format("delta").mode("overwrite").saveAsTable("fintech_fraud_risk.gold.dim_date")

In [0]:
assert fact_transactions.count() == spark.table("fintech_fraud_risk.silver.transactions").count(), "Row count mismatch — fact table has fanned out or dropped rows"

In [0]:
spark.table("fintech_fraud_risk.silver.transactions").printSchema()
spark.table("fintech_fraud_risk.silver.transactions").select("transaction_timestamp").show(5)